# Agent Architecture: Reflection + Light Tool Use

This notebook walks through the core agent pattern used in the project.

The pipeline is intentionally simple:

1. Analyzer
2. Critic
3. Reviser

The Analyzer can call a tiny statistical tool layer before producing its first structured diagnosis. The Critic then checks for missing logic, weak evidence, and overconfident claims. Finally, the Reviser integrates both views into a cleaner answer.

## Why this counts as an agentic pattern

The project demonstrates three core ideas often used in modern LLM systems:

- **Reflection**: the model critiques its own earlier work instead of trusting the first draft.
- **ReAct-style tool use**: the model chooses a small number of tools when statistical evidence would strengthen the diagnosis.
- **Structured outputs**: every stage returns JSON-compatible fields, which makes downstream evaluation and debugging much easier.

This is deliberately lighter than a fully general autonomous agent loop. The point is clarity and inspectability.

In [ ]:
from rct_diagnosis_agent.agent import build_default_agent
from rct_diagnosis_agent.data import SyntheticExperimentGenerator

generator = SyntheticExperimentGenerator(seed=11)
experiment = generator.generate("exp_walkthrough", failure_modes=["srm", "imbalance", "low_power"])
agent = build_default_agent()

## Step 1: inspect the experiment summary

Before we run the full agent, it is helpful to inspect the structured input. This is exactly the object handed to the Analyzer.

In [ ]:
experiment.to_prompt_dict()

## Step 2: run the end-to-end pipeline

When LangSmith tracing is enabled, this run should create separate spans for:

- analyzer planning
- tool execution
- analyzer output
- critic output
- reviser output

That makes the internal reasoning path much easier to inspect than a single opaque completion.

In [ ]:
result = agent.run(experiment)
result.model_dump()

## Why structured outputs matter

The Analyzer, Critic, and Reviser all return schema-constrained data. That creates several benefits:

- easier scoring against ground truth
- less brittle notebook analysis
- cleaner traces in LangSmith
- a tighter contract between stages

In a production system we would likely add stronger validation and versioning. For an educational system, these schema boundaries already provide a lot of value.

## A useful reading strategy

When you inspect a trace, read it in this order:

1. Which tools were selected and why?
2. Did the Analyzer ground its claims in the tool outputs?
3. Did the Critic spot missing checks or unsupported jumps?
4. Did the Reviser actually improve the diagnosis?

That order keeps the focus on evidence, not on surface-level wording.